# CallSense: Arabic Telecom Call-Center Intelligence

**Pipeline:** Whisper small (speech-to-text) → bge-m3 (retrieval) → Qwen2.5-7B-Instruct (intent, escalation, summary, resolution) → Gradio web app with a public link.

### How to run
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive create a folder named `CallSense` and upload `data/Whisper_Small_Baseline_Results_with_Resolution.xlsx` into it.
3. Run the cells from top to bottom (restart the session once, right after the install cell).
4. The last app cell prints a public `https://....gradio.live` link. Keep the notebook running while people use it.

> The Arabic notes below are the team's working notes. See the repository README and `docs/REPORT.md` for the full English documentation.


# CallSense v3

<div dir="rtl" style="text-align: right;">

**الخط:** تسجيل صوتي من المتصفح ← **Whisper** يكتب النص  ← **bge-m3** يجيب أقرب مكالمات مصنّفة  

 ← **Qwen** يحدد التصنيف والتصعيد ثم يكتب الملخص والحل  ← تظهر النتيجة في صفحة  Gradio برابط عام.

مودل TF-IDF (المصنّف\ Classifer) بنيناه وقيّمناه، وما عاد نحمّله في الدفتر. ننقل القرار النهائي لتصنيف المكالمة والتصعيد إلى  Qwen.

**قبل البدء:**
1. Runtime ← Change runtime type ← **T4 GPU**
2. في Google Drive، مجلد **`CallSense`** فيه الملف: `Whisper\_Small\_Baseline\_Results\_with\_Resolution.xlsx`

**الترتيب:** خلية 1 ← **Restart session** ← بقية الخلايا بالترتيب.
</div>

## 1) تثبيت المكتبات


In [1]:
!pip install -q "pandas==2.2.3" sentence-transformers openpyxl
!pip install -q -U bitsandbytes accelerate openai-whisper "gradio>=5,<6"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
python-fasthtml 0.14.14 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.


## 2) ربط Drive والتأكد من الملف

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")

STORE_DIR = "/content/drive/MyDrive/CallSense"
KB_XLSX   = os.path.join(STORE_DIR, "Whisper_Small_Baseline_Results_with_Resolution.xlsx")
os.makedirs(STORE_DIR, exist_ok=True)

print("✅" if os.path.exists(KB_XLSX) else "❌ ناقص:", KB_XLSX)
assert os.path.exists(KB_XLSX), "ارفعوا الملف لمجلد CallSense في Drive ثم أعيدوا الخلية"

Mounted at /content/drive
✅ /content/drive/MyDrive/CallSense/Whisper_Small_Baseline_Results_with_Resolution.xlsx


## 3) قاعدة المكالمات المصنّفة (٨٠ مكالمة + الحلول المعتمدة)
هذي القاعدة هي اللي يتعلم منها Qwen أسلوب التصنيف، وهي نفسها اللي يبحث فيها عن حلول مشابهة.

In [3]:
import pandas as pd, numpy as np

df = pd.read_excel(KB_XLSX, sheet_name="Results")

df["intent"] = df["intent"].str.strip().str.lower()                          # توحيد أسماء الفئات
for c in ["reference_transcript", "resolution", "review_status"]:
    df[c] = df[c].fillna("").astype(str).str.replace("\\n", "\n")            # \n النصية تصير سطر جديد
df.loc[df["review_status"].str.strip() == "مرفوض", "resolution"] = ""        # الحل المرفوض ما يدخل القاعدة

kb = df.rename(columns={"whisper_prediction": "call_text"})
kb = kb[["call_id", "intent", "escalation", "call_text", "reference_transcript", "resolution"]].reset_index(drop=True)
kb["call_text"] = kb["call_text"].astype(str)

print(len(kb), "مكالمة | فيها حل معتمد:", int((kb["resolution"].str.strip() != "").sum()))
print(kb.intent.value_counts())

80 مكالمة | فيها حل معتمد: 0
intent
information     19
technical       14
cancellation    13
billing         12
complaint       12
out_of_scope    10
Name: count, dtype: int64


In [4]:
import os
from google.colab import drive
drive.mount("/content/drive")

STORE_DIR = "/content/drive/MyDrive/CallSense"
LOG_PATH = os.path.join(STORE_DIR, "results_log.csv")
APPROVED_PATH = os.path.join(STORE_DIR, "approved_kb.csv")
os.makedirs(STORE_DIR, exist_ok=True)

KB_COLS = ["call_id", "intent", "escalation", "call_text", "reference_transcript", "resolution"]
if os.path.exists(APPROVED_PATH):
    approved = pd.read_csv(APPROVED_PATH, dtype=str).fillna("")
    kb = pd.concat([kb, approved[KB_COLS]], ignore_index=True)
    print("انضاف للقاعدة", len(approved), "حل معتمد")
kb = kb.fillna("").reset_index(drop=True)
print("حجم القاعدة:", len(kb), "| السجل:", LOG_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
حجم القاعدة: 80 | السجل: /content/drive/MyDrive/CallSense/results_log.csv


## 4) تحميل الموديلات: Whisper + bge-m3 + Qwen
تاخذ 5–10 دقائق أول مرة.

In [5]:
import torch, whisper
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, BitsAndBytesConfig

# 1) Whisper small: الصوت إلى نص
whisper_model = whisper.load_model("small", device="cuda")

# 2) bge-m3: يجيب أقرب مكالمات مصنّفة   3) Qwen: التصنيف والتصعيد والملخص والحل
EMB = "BAAI/bge-m3"
LLM = "Qwen/Qwen2.5-7B-Instruct"

emb_model = SentenceTransformer(EMB)
gen = pipeline("text-generation", model=LLM, device_map="auto",
               model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_4bit=True)})

kb_emb = emb_model.encode(kb["call_text"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
print("جاهز: whisper-small +", EMB, "+", LLM)

100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 81.0MiB/s]


جاهز: whisper-small + BAAI/bge-m3 + Qwen/Qwen2.5-7B-Instruct


## 5) دوال المودلات
**Whisper**
<div dir="rtl" style="text-align: right;">

يستخدم إعدادات `tuned` افتراضيًا (نفس Whisper small بدون fine-tuning)، وللرجوع للأصل: `WHISPER_MODE = "baseline"`.

وفيه حماية من تكرار الكلمات (repetition guard): لو علق Whisper وكرر كلمة، يعيد التفريغ تلقائيًا بإعداد إنقاذ.

In [6]:
import os, re, zlib

WHISPER_PROMPT = "مكالمة خدمة عملاء شركة اتصالات سعودية: باقة، فاتورة، راوتر، إنترنت، شريحة، إلغاء الاشتراك، شكوى، فني."
WHISPER_MODES = {
    "baseline": dict(language="ar", task="transcribe", fp16=True),
    "tuned": dict(language="ar", task="transcribe", fp16=True, temperature=0.0, beam_size=5,
                  condition_on_previous_text=False, initial_prompt=WHISPER_PROMPT),
    # للإنقاذ لما يعلق Whisper ويكرر كلمة: يسمح بدرجات حرارة بديلة (fallback) عشان يكسر التكرار
    "rescue": dict(language="ar", task="transcribe", fp16=True, temperature=(0.0, 0.2, 0.4, 0.6, 0.8),
                   compression_ratio_threshold=2.4, condition_on_previous_text=False),
}
WHISPER_MODE = "tuned"

def looks_repetitive(text):
    """يكشف تكرار Whisper: نفس الكلمة ٦ مرات متتالية، أو نص مضغوط بشكل غير طبيعي (نفس مقياس Whisper الداخلي)."""
    words = text.split()
    run = 1
    for a, b in zip(words, words[1:]):
        run = run + 1 if a == b else 1
        if run >= 6:
            return True
    raw = text.encode("utf-8")
    return len(words) >= 12 and len(raw) / max(len(zlib.compress(raw)), 1) > 2.4

def collapse_repeats(text, keep=2):
    """آخر حل: لو بقي تكرار، نختصر أي كلمة مكررة ٣ مرات أو أكثر إلى مرتين."""
    return re.sub(r"(\S+)(?:\s+\1){%d,}" % keep, lambda m: " ".join([m.group(1)] * keep), text)

def transcribe_audio(audio_path, mode=None):
    """
    Takes an audio file path and returns its Arabic transcription.
    mode: "tuned" (default) or "baseline". If Whisper loops (repeats a word), it retries automatically.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    mode = mode or WHISPER_MODE
    run = lambda m: whisper_model.transcribe(audio_path, **WHISPER_MODES[m])["text"].strip()
    text = run(mode)
    if mode == "tuned" and WHISPER_PROMPT[:25] in text:      # Whisper كرر نص التوجيه نفسه
        text = run("baseline")
    if looks_repetitive(text):                               # Whisper علق وكرر كلمة
        for m in ("rescue", "baseline"):
            alt = run(m)
            if not looks_repetitive(alt):
                text = alt
                break
        else:
            text = collapse_repeats(text)
    return text

print("✅ transcribe_audio() is ready! | mode:", WHISPER_MODE)

✅ transcribe_audio() is ready! | mode: tuned


**البحث عن مكالمات مشابهة**

In [7]:
INTENT_AR = {"billing": "الفواتير", "technical": "مشكلة تقنية", "cancellation": "إلغاء",
             "complaint": "شكوى", "information": "استفسار", "out_of_scope": "خارج النطاق"}

OOS_REPLY = ("عذرًا، أنا مساعد متخصص في خدمات الاتصالات فقط، مثل الفواتير والباقات والمشاكل التقنية والإلغاء. "
             "ما أقدر أساعدك في هذا الطلب، لكن إذا عندك أي استفسار يخص خدمات الاتصالات أنا حاضر.")

def norm_intent(v):
    return str(v).strip().lower().replace(" ", "_")

def to_bool(v):
    return str(v).strip().lower() in ("1", "1.0", "true", "yes", "y", "نعم")

def retrieve(text, intent=None, exclude_id=None, k=3):
    q = emb_model.encode(text, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(q, kb_emb)[0]
    rank = sims.clone()
    if intent is not None:
        same = torch.tensor((kb.intent == intent).values, device=rank.device)
        rank = rank + same.float()                       # نفس الفئة أولًا، والباقي يكمّل لو ما كفى
    for i in kb.index[kb.call_id == exclude_id]:
        rank[i] = float("-inf")                          # لا يسترجع نفس المكالمة
    idx = rank.topk(k).indices.tolist()
    return idx, [round(float(sims[i]), 3) for i in idx]

def vote_out_of_scope(text, exclude_id=None, k=3):
    idx, _ = retrieve(text, intent=None, exclude_id=exclude_id, k=k)
    return sum(kb.intent[i] == "out_of_scope" for i in idx) > k / 2

<div dir="rtl" style="text-align: right;">

**حراس الجودة** (جديد): يمنعون ظهور حروف صينية أو أي لغة غريبة في المخرجات، ويطلّعون أرقام المكالمات بدون أقواس (هذا كان سبب تنبيه "يحتاج مراجعة" الكاذب)، ويشيلون الترقيم المكرر من الخطوات. لو طلع مخرج بلغة غلط، يعيد المحاولة مرة وحدة بتعليمات أشد.

</div>

In [8]:
import json, re, time

# ============ حراس الجودة: لغة المخرجات + أرقام المكالمات + ترقيم الخطوات ============
# أي حرف صيني/ياباني/كوري/روسي/عبري/هندي/تايلندي يُعتبر خطأ ولا يوصل للشاشة
BAD_SCRIPT = re.compile(r"[\u0400-\u04FF\u0590-\u05FF\u0900-\u0DFF\u0E00-\u0EFF\u1100-\u11FF"
                        r"\u3000-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF\uFF00-\uFFEF]")
AR_LETTER  = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
LAT_LETTER = re.compile(r"[A-Za-z]")
LANG_NOTE  = "تنبيه مهم: اكتب كل النصوص بالعربية فقط. ممنوع استخدام الصينية أو أي لغة أخرى. أعد JSON فقط."

def text_ok(s, min_ar=0.5):
    """نص سليم = بدون حروف لغات غريبة، وأغلب حروفه عربية (لو فيه حروف أصلًا)."""
    s = str(s or "")
    if BAD_SCRIPT.search(s):
        return False
    ar, la = len(AR_LETTER.findall(s)), len(LAT_LETTER.findall(s))
    return (ar + la == 0) or ar / (ar + la) >= min_ar

def strings_in(x):
    if isinstance(x, str):
        yield x
    elif isinstance(x, dict):
        for v in x.values():
            yield from strings_in(v)
    elif isinstance(x, (list, tuple)):
        for v in x:
            yield from strings_in(v)

def strip_bad(s):
    return BAD_SCRIPT.sub("", str(s or "")).strip()

# أرقام المكالمات: Qwen يكتبها أحيانًا [CS003] بأقواس، فنطلّع الرقم نفسه قبل أي مقارنة
ID_RE = re.compile(r"CS\d+|R\d{8}-\d{6}-[0-9a-fA-F]{4}", re.I)

def norm_ids(v):
    items = v if isinstance(v, (list, tuple)) else [v]
    out = []
    for s in items:
        for m in ID_RE.findall(str(s)):
            if m[:2].upper() == "CS":
                m = m.upper()
            elif m[0] in "rR":
                m = "R" + m[1:]
            if m not in out:
                out.append(m)
    return out

# ترقيم الخطوات: الصفحة ترقّم بنفسها، فنشيل أي ترقيم كتبه الموديل ("1." أو "2)" أو "٣-")
NUM_RE = re.compile(r"^\s*\(?(?:\d+|[\u0660-\u0669]+)\)?\s*[\.\)\-\u2013:\u060C]\s*")

def strip_num(s):
    return NUM_RE.sub("", str(s)).strip()

def parse_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def generate_json(messages, max_new_tokens, validate):
    """يولّد JSON. لو طلع خربان أو فيه لغة غلط، يعيد مرة وحدة بتعليمات أشد. يرجع (data, raw, أُعيدت_المحاولة)."""
    data, raw = None, ""
    for attempt in range(2):
        msgs = list(messages)
        if attempt == 1:
            msgs[-1] = {"role": msgs[-1]["role"], "content": msgs[-1]["content"] + "\n\n" + LANG_NOTE}
        out = gen(msgs, max_new_tokens=max_new_tokens, do_sample=False)      # بدون repetition_penalty (يسبب انجراف للصيني)
        raw = out[0]["generated_text"][-1]["content"]
        data = parse_json(raw)
        if isinstance(data, dict) and validate(data):
            return data, raw, attempt == 1
    return (data if isinstance(data, dict) else None), raw, True

print("✅ guards ready")

✅ guards ready


**Qwen: الملخص والحل**

In [9]:
SYS_DOC = """أنت خبير جودة في مركز خدمة عملاء شركة اتصالات سعودية. مهمتك كتابة ملخص المكالمة وحلها بشكل احترافي ليكون مرجعًا للموظفين والمشرفين.
القواعد:
- الفئة وقرار التصعيد يأتيان من نظام التصنيف. اعتمدهما ولا تغيّرهما.
- إذا رأيت أن الفئة لا تطابق محتوى المكالمة بوضوح، اذكر ذلك في intent_check مع السبب.
- اعتمد على نص المكالمة أولًا، ثم على المكالمات المشابهة المعطاة.
- إذا طبّق الموظف حلًا واضحًا، وثّقه كما حدث بدون إضافة معلومات غير موجودة.
- إذا كان الحل غير واضح أو ناقصًا، اقترح حلًا احترافيًا عمليًا مستفيدًا من المكالمات المشابهة.
- إذا كان التصعيد مطلوبًا، اجعل آخر خطوة في الحل هي التسليم للمشرف، واكتب escalation_handoff: المشكلة، وما تمت تجربته، والمطلوب من المشرف.
- إذا لم يكن التصعيد مطلوبًا، اترك escalation_handoff فارغًا.
- لا تخترع أرقامًا أو أسعارًا أو سياسات غير مذكورة في النصوص.
- اكتب بالعربية الفصحى المختصرة فقط، ولا تستخدم أي لغة أخرى.
- لا تضع أرقامًا في بداية الخطوات، اكتب كل خطوة كجملة مستقلة.
- في sources اكتب أرقام المكالمات فقط مثل CS003 بدون أقواس.
- أعد JSON فقط بدون أي نص قبله أو بعده."""

SCHEMA = """{
 "summary": "ملخص المشكلة في سطرين",
 "solution_status": "واضح" أو "جزئي" أو "غير موجود",
 "applied_solution": "ما فعله الموظف فعلًا، أو نص فارغ",
 "recommended_solution": ["خطوة", "خطوة"],
 "escalation_handoff": "ملاحظة التسليم للمشرف إذا كان التصعيد مطلوبًا، وإلا نص فارغ",
 "intent_check": "متوافق" أو "غير متوافق: السبب",
 "sources": ["أرقام المكالمات المشابهة التي استفدت منها فقط"]
}"""

FALLBACK_SUMMARY = "تعذّر توليد ملخص عربي سليم، يرجى مراجعة نص المكالمة."
STATUSES = ("واضح", "جزئي", "غير موجود")

def _digest_ok(d):
    keys = ("summary", "applied_solution", "recommended_solution", "escalation_handoff", "intent_check")
    return all(text_ok(s) for s in strings_in({k: d.get(k) for k in keys}))

def generate_digest(call_text, intent, escalation, call_id=None, k=3, topic=None, scope_confirmed=False):
    """المدخلات من خطوة التصنيف: intent و escalation. المخرجات: dict فيه الملخص والحل والتنبيهات."""
    intent, esc = norm_intent(intent), to_bool(escalation)
    flags = []
    vote_oos = vote_out_of_scope(call_text, exclude_id=call_id, k=k)

    if intent == "out_of_scope":
        if not vote_oos and not scope_confirmed:
            flags.append("التصنيف خارج النطاق لكن المكالمات المشابهة من خدمات الاتصالات")
        t = strip_bad(topic)
        summary = "طلب خارج نطاق خدمات الاتصالات" + (f": {t}" if t and text_ok(t) else "")
        return {"call_id": call_id, "intent": intent, "escalation": esc,
                "summary": summary, "solution_status": "خارج النطاق",
                "applied_solution": "", "recommended_solution": [OOS_REPLY], "escalation_handoff": "",
                "intent_check": "متوافق" if (vote_oos or scope_confirmed) else "يحتاج مراجعة", "sources": [], "retrieved": [],
                "confidence": "", "flags": flags, "parse_ok": True, "raw": ""}

    if vote_oos:
        flags.append("المكالمات المشابهة تشير إلى أن الطلب قد يكون خارج النطاق")

    idx, sims = retrieve(call_text, intent=intent, exclude_id=call_id, k=k)
    retrieved = [kb.call_id[i] for i in idx]
    def case_text(i):
        t = f"[{kb.call_id[i]}] ({INTENT_AR.get(kb.intent[i], kb.intent[i])})\n{kb.reference_transcript[i]}"
        if str(kb.resolution[i]).strip():
            t += f"\nالحل المعتمد لهذه المكالمة:\n{kb.resolution[i]}"
        return t
    ctx = "\n\n".join(case_text(i) for i in idx)
    user = (f"معلومات من نظام التصنيف (اعتمدها كما هي):\n"
            f"- الفئة: {INTENT_AR.get(intent, intent)}\n"
            f"- التصعيد: {'مطلوب' if esc else 'غير مطلوب'}\n\n"
            f"نص المكالمة [{call_id or 'NEW'}]:\n{call_text}\n\n"
            f"مكالمات مشابهة سابقة:\n{ctx}\n\nأعد JSON بهذا الشكل:\n{SCHEMA}")
    data, raw, retried = generate_json([{"role": "system", "content": SYS_DOC}, {"role": "user", "content": user}],
                                       700, _digest_ok)
    d = data or {}
    if data is None:
        flags.append("الموديل ما رجع JSON صحيح")

    # المصادر: نطلّع الأرقام (بدون أقواس) ونقارنها بالمكالمات اللي استرجعناها فعلًا
    cited = norm_ids(d.get("sources") if isinstance(d.get("sources"), (list, str)) else [])
    if any(s not in retrieved for s in cited):
        flags.append("استشهد بمكالمات غير مسترجعة")
    sources = [s for s in cited if s in retrieved]

    # تنظيف اللغة والترقيم
    lang_bad = data is not None and not _digest_ok(d)
    steps = d.get("recommended_solution", [])
    if isinstance(steps, str):
        steps = [x for x in steps.split("\n") if x.strip()]
    elif not isinstance(steps, list):
        steps = []
    clean_steps = [s for s in (strip_num(strip_bad(x)) for x in map(str, steps)) if s and text_ok(s)]
    summary = strip_bad(d.get("summary", ""))
    if not (summary and text_ok(summary)):
        summary = FALLBACK_SUMMARY if data is not None or lang_bad else ""
    applied = strip_bad(d.get("applied_solution", ""))
    applied = applied if text_ok(applied) else ""
    handoff = strip_bad(d.get("escalation_handoff", ""))
    handoff = handoff if text_ok(handoff) else ""
    check = strip_bad(d.get("intent_check", ""))
    check = check if text_ok(check) else ""
    if lang_bad:
        flags.append("مخرجات الموديل لم تكن بالعربية السليمة")
    elif data is not None and not clean_steps:
        flags.append("لم يُولَّد حل واضح")

    status = d.get("solution_status", "")
    if status not in STATUSES:
        status = "جزئي" if clean_steps else "غير موجود"
    if check.startswith("غير"):
        flags.append("الموديل يشك في الفئة: " + check)
    if esc and not handoff:
        flags.append("التصعيد مطلوب لكن ما فيه ملاحظة تسليم")

    return {"call_id": call_id, "intent": intent, "escalation": esc,
            "summary": summary, "solution_status": status,
            "applied_solution": applied, "recommended_solution": clean_steps,
            "escalation_handoff": handoff, "intent_check": check,
            "sources": sources, "retrieved": retrieved, "confidence": "",
            "flags": flags, "parse_ok": data is not None, "raw": "" if data else raw}

print("Ready")

Ready


**Qwen: التصنيف والتصعيد** (الأحدث)
<div dir="rtl" style="text-align: right;">
يشوف أقرب ٣ مكالمات مصنّفة من القاعدة ويقرر. لو رجّع شي غير صالح، نرجع لتصويت المكالمات الأقرب.

التصنيف الحين يبدأ بسؤال: **هل المكالمة أصلًا عن خدمات الاتصالات؟** لو لا (مطعم، بنك، سؤال عام...) فهي `out_of_scope` والكود يفرض ذلك، ولا يعتمد على مزاج المودل
</div>

In [10]:
# ============ تصنيف المكالمة والتصعيد بواسطة Qwen: يبدأ بسؤال النطاق قبل أي تصنيف ============
SYS_CLS = """أنت مصنّف مكالمات في مركز خدمة عملاء شركة اتصالات سعودية. اتبع الخطوات بالترتيب:

الخطوة 1: ما موضوع المكالمة فعليًا؟ اكتبه في topic بكلمات قليلة (مثل: "تأخر طلب من مطعم"، "انقطاع الإنترنت"، "فاتورة مكررة").

الخطوة 2: هل الطلب يخص خدمات شركة الاتصالات نفسها؟ (telecom_related)
- true فقط إذا كان الطلب عن: خط جوال أو شريحة، باقة أو عرض، الإنترنت أو الراوتر أو الإشارة أو الشبكة، فاتورة أو دفع أو رصيد، إلغاء أو نقل اشتراك، أو شكوى من شركة الاتصالات نفسها أو موظفيها أو فروعها أو مواعيد فنييها.
- false إذا كان الطلب عن أي جهة أو خدمة أخرى: مطعم، توصيل، متجر، بنك، مستشفى، شركة طيران، مدرسة، أو سؤال عام، أو ترجمة، أو طلب شخصي.
  وهذا ينطبق حتى لو كان العميل منزعجًا أو ينتظر طويلًا، وحتى لو بدأ الموظف المكالمة بعبارة "خدمة العملاء". الشكوى من مطعم أو مندوب توصيل أو بنك ليست شكوى لشركة الاتصالات.

الخطوة 3: إذا كانت telecom_related = false فالفئة out_of_scope والتصعيد 0.

الخطوة 4: إذا كانت true، اختر فئة واحدة والأساس هو طلب العميل الرئيسي:
- billing: الفواتير والمبالغ والرسوم والخصومات والاسترجاع والدفع.
- technical: مشاكل تقنية في الإنترنت أو الشبكة أو الإشارة أو الراوتر أو الخدمة.
- cancellation: العميل يطلب إلغاء اشتراك أو خدمة أو إيقافها، حتى لو كان السبب مشكلة أخرى.
- complaint: شكوى من تجربة أو سلوك موظف أو تأخر أو إخلاف موعد أو سوء خدمة، بدون طلب تقني أو مالي أو إلغاء محدد.
- information: استفسار عام عن باقة أو سعر أو خدمة أو عرض أو إجراء، بدون مشكلة.

التصعيد (escalation = 1) مطلوب إذا انطبق أحد هذه:
- المشكلة تكررت أو سبق للعميل التبليغ عنها ولم تُحل (بلاغات أُغلقت والمشكلة باقية، موعد تأجل أكثر من مرة).
- وعد أو التزام من الشركة لم يُنفَّذ (مثل خصم وُعد به ولم يُطبَّق).
- العميل منزعج بوضوح، أو يهدد بالإلغاء، أو فقد ثقته بالخدمة.
- الحل يحتاج صلاحية أعلى من الموظف (تعويض، زيارة فني، استثناء).
غير ذلك escalation = 0. مكالمات information وout_of_scope لا تحتاج تصعيدًا.

القواعد:
- اعتمد على المكالمة نفسها. الأمثلة المرفقة لتوضيح الأسلوب فقط، ولا تنسخ تصنيفها.
- اكتب topic وreason بالعربية فقط، ولا تستخدم أي لغة أخرى.
- أعد JSON فقط بهذا الترتيب: topic ثم telecom_related ثم intent ثم escalation ثم reason."""

SCOPE_EXAMPLES = """أمثلة توضيحية ثابتة:

[مثال 1] العميل يقول إنه طلب وجبة من مطعم قبل ساعة ولم تصل، والموظف يعتذر ويتواصل مع المندوب.
{"topic": "تأخر توصيل طلب من مطعم", "telecom_related": false, "intent": "out_of_scope", "escalation": 0, "reason": "المكالمة عن طلب مطعم وليست عن خدمات الاتصالات"}

[مثال 2] العميل يقول إن بطاقته البنكية أُوقفت ويسأل كيف يفعّلها.
{"topic": "إيقاف بطاقة بنكية", "telecom_related": false, "intent": "out_of_scope", "escalation": 0, "reason": "الطلب عن خدمة بنكية وليس عن شركة الاتصالات"}

[مثال 3] العميل يقول إن الإنترنت ينقطع كل ساعة وجرّب إعادة تشغيل الراوتر، والموظف سيرسل فنيًا.
{"topic": "انقطاع متكرر في الإنترنت", "telecom_related": true, "intent": "technical", "escalation": 0, "reason": "مشكلة تقنية في خدمة الإنترنت لم تتكرر عبر بلاغات سابقة"}"""

TRUE_SET  = ("true", "1", "yes", "نعم")
FALSE_SET = ("false", "0", "no", "لا")

def _fmt_example(i):
    t = str(kb.reference_transcript[i]).strip()
    if len(t) > 600:
        t = t[:600] + " ..."
    return f"[{kb.call_id[i]}] الفئة: {norm_intent(kb.intent[i])} | التصعيد: {1 if to_bool(kb.escalation[i]) else 0}\n{t}"

def classify_with_qwen(call_text, call_id=None, k=3):
    """يرجع الفئة والتصعيد. يسأل أول: هل المكالمة عن خدمات الاتصالات؟ والكود يفرض النتيجة لو الجواب لا."""
    idx, _ = retrieve(call_text, intent=None, exclude_id=call_id, k=k)
    examples = "\n\n".join(_fmt_example(i) for i in idx)
    user = (f"{SCOPE_EXAMPLES}\n\nأمثلة من قاعدتنا المصنّفة (لتوضيح أسلوب التصنيف فقط):\n{examples}\n\n"
            f"المكالمة المطلوب تصنيفها [{call_id or 'NEW'}]:\n{call_text}\n\n"
            'أعد JSON فقط بهذا الشكل: {"topic": "...", "telecom_related": true أو false, "intent": "...", '
            '"escalation": 0 أو 1, "reason": "سبب في جملة قصيرة"}')
    ok = lambda d: text_ok(d.get("topic", "")) and text_ok(d.get("reason", ""))
    d, raw, retried = generate_json([{"role": "system", "content": SYS_CLS}, {"role": "user", "content": user}], 160, ok)
    d = d or {}

    intent = norm_intent(d.get("intent", ""))
    tele = str(d.get("telecom_related", "")).strip().lower()
    esc_raw = str(d.get("escalation", "")).strip().lower()
    topic = strip_bad(d.get("topic", ""))
    reason = strip_bad(d.get("reason", ""))
    topic = topic if text_ok(topic) else ""
    reason = reason if text_ok(reason) else ""

    valid = intent in INTENT_AR and tele in TRUE_SET + FALSE_SET and esc_raw in TRUE_SET + FALSE_SET
    if valid and tele in FALSE_SET:                                   # الجواب "مو عن الاتصالات" ← خارج النطاق دائمًا
        intent, esc, method = "out_of_scope", False, "qwen"
    elif valid and tele in TRUE_SET and intent == "out_of_scope":     # تناقض ← نرجع لتصويت أقرب المكالمات
        valid = False
    elif valid:
        esc, method = to_bool(esc_raw), "qwen"

    if not valid:                                                     # احتياط: تصويت أقرب المكالمات المصنّفة
        votes = [norm_intent(kb.intent[i]) for i in idx]
        intent = max(set(votes), key=votes.count)
        esc = sum(to_bool(kb.escalation[i]) for i in idx) * 2 >= len(idx)
        method = "knn-fallback"
        reason = "Fallback: nearest labeled calls | بالاعتماد على أقرب المكالمات المصنّفة"
    if intent in ("information", "out_of_scope"):                     # حسب داتاكم: هذي الفئات ما تُصعَّد
        esc = False
    if intent == "out_of_scope" and not reason:
        reason = "الطلب لا يخص خدمات الاتصالات"
    return {"intent": intent, "escalation": int(esc), "reason": reason, "topic": topic, "method": method,
            "telecom_related": intent != "out_of_scope", "retried": retried, "neighbors": [kb.call_id[i] for i in idx]}

print("✅ classify_with_qwen() Ready")

✅ classify_with_qwen() Ready


<div dir="rtl" style="text-align: right;">

**التقرير النصي + تجربة سريعة** على مكالمة CS003 من القاعدة (نص فقط، بدون صوت). لو طلع تصنيف وتقرير عربي مرتب، Qwen شغّال.

In [11]:
ESC_AR = {True: "مطلوب", False: "غير مطلوب"}

def format_report(res):
    lines = ["📋 تقرير المكالمة", "",
             f"التصنيف: {INTENT_AR.get(res['intent'], res['intent'])}",
             f"التصعيد: {ESC_AR[bool(res['escalation'])]}", "",
             "الملخص:", str(res["summary"]), ""]
    if res["solution_status"] == "خارج النطاق":
        lines += ["الرد:", OOS_REPLY]
    else:
        lines.append(f"الحل ({res['solution_status']}):")
        steps = res["recommended_solution"] if isinstance(res["recommended_solution"], list) else [res["recommended_solution"]]
        lines += [f"{n}. {s}" for n, s in enumerate(steps, 1)]
        if res["applied_solution"]:
            lines += ["", "ما طبّقه الموظف:", str(res["applied_solution"])]
        if res["escalation_handoff"]:
            lines += ["", "ملاحظة التسليم للمشرف:", str(res["escalation_handoff"])]
        if res["sources"]:
            lines += ["", "مبني على مكالمات: " + "، ".join(map(str, res["sources"]))]
    if res["flags"]:
        lines += ["", "⚠️ يحتاج مراجعة:"] + [f"- {f}" for f in res["flags"]]
    return "\n".join(lines)

# تجربة سريعة بالسلسلة النصية كاملة (بدون صوت): تصنيف Qwen ثم الملخص والحل
call_text = kb[kb.call_id == "CS003"].iloc[0].call_text
cls = classify_with_qwen(call_text, call_id="CS003")
res = generate_digest(call_text, cls["intent"], cls["escalation"], call_id="CS003",
                      topic=cls["topic"], scope_confirmed=(cls["method"] == "qwen"))
print("التصنيف:", cls, "\n")
print(format_report(res))

التصنيف: {'intent': 'billing', 'escalation': 0, 'reason': 'شكوى حول زيادة مبلغ الفاتورة دون تنبيه مسبق', 'topic': 'زيادة مبلغ الفاتورة', 'method': 'qwen', 'telecom_related': True, 'retried': False, 'neighbors': ['CS001', 'CS002', 'CS066']} 

📋 تقرير المكالمة

التصنيف: الفواتير
التصعيد: غير مطلوب

الملخص:
الفاتورة أعلى من المعتاد بسبب استهلاك بيانات إضافية، والعميل يريد تفاصيل باقة البيانات الجديدة.

الحل (جزئي):
1. أرسل تفاصيل باقة البيانات الجديدة برسالة نصية للعميل.
2. إرشاد العميل لاستخدام تطبيق الشركة لمشاهدة تفاصيل الاستهلاك.
3. إبلاغ العميل بإرسال إشعار عند وصول الاستهلاك إلى 80% من الحد المتاح.

مبني على مكالمات: CS001، CS002


**السجل والمراجعة**: كل مكالمة تنحفظ في `results_log.csv` على Drive. الاعتماد يقبل تصحيح التصنيف والتصعيد.

وتحت فيه دوال **الحذف والتعديل والنسخ الاحتياطي** (من الدفتر فقط)، مشروحة في آخر الدفتر.

In [12]:
import uuid, shutil
from datetime import datetime

LOG_COLS = ["record_id", "timestamp", "call_text", "intent", "escalation", "summary", "solution_status",
            "applied_solution", "recommended_solution", "escalation_handoff", "intent_check", "sources",
            "retrieved", "confidence", "flags", "review_status", "final_resolution", "reviewer_notes",
            "intent_reason", "intent_method"]

def j(v):
    return "\n".join(map(str, v)) if isinstance(v, list) else str(v or "")

def _migrate_log():
    """لو ملف السجل قديم (بدون الأعمدة الجديدة) نضيفها بدون ما نفقد أي مكالمة."""
    if not os.path.exists(LOG_PATH):
        return
    old = pd.read_csv(LOG_PATH, dtype=str).fillna("")
    if list(old.columns) == LOG_COLS:
        return
    for c in LOG_COLS:
        if c not in old.columns:
            old[c] = ""
    old[LOG_COLS].to_csv(LOG_PATH, index=False, encoding="utf-8-sig")

def load_log():
    if not os.path.exists(LOG_PATH):
        return pd.DataFrame(columns=LOG_COLS)
    d = pd.read_csv(LOG_PATH, dtype=str).fillna("")
    for c in LOG_COLS:
        if c not in d.columns:
            d[c] = ""
    return d[LOG_COLS]

def save_result(res, call_text):
    _migrate_log()
    rid = "R" + datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:4]
    row = {"record_id": rid, "timestamp": datetime.now().isoformat(timespec="seconds"), "call_text": call_text,
           "intent": res["intent"], "escalation": int(res["escalation"]),
           **{k: j(res.get(k, "")) for k in ["summary", "solution_status", "applied_solution", "recommended_solution",
                                             "escalation_handoff", "intent_check", "sources", "retrieved", "confidence", "flags"]},
           "review_status": "", "final_resolution": "", "reviewer_notes": "",
           "intent_reason": str(res.get("intent_reason", "")), "intent_method": str(res.get("intent_method", ""))}
    pd.DataFrame([row], columns=LOG_COLS).to_csv(LOG_PATH, mode="a", index=False,
                                                 header=not os.path.exists(LOG_PATH), encoding="utf-8-sig")
    return rid

def pending():
    log = load_log()
    return log[log.review_status == ""][["record_id", "timestamp", "intent", "escalation", "intent_reason",
                                         "summary", "solution_status", "flags"]]

def _set_review(rid, status, final="", notes=""):
    log = load_log()
    m = log.record_id == rid
    if not m.any():
        raise ValueError(f"ما لقيت النتيجة {rid}")
    log.loc[m, ["review_status", "final_resolution", "reviewer_notes"]] = [status, final, notes]
    log.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    return log[m].iloc[0]

def approve(rid, edited_resolution="", notes="", intent=None, escalation=None):
    """اعتماد الحل. لو تصنيف Qwen أو تصعيده غلط، صحّحيه هنا: approve(rid, intent="complaint", escalation=1)"""
    global kb, kb_emb
    log = load_log()
    m = log.record_id == rid
    if not m.any():
        raise ValueError(f"ما لقيت النتيجة {rid}")
    row = log[m].iloc[0]
    if row.review_status in ("معتمد", "معدّل"):
        print(f"{rid} معتمد من قبل")
        return
    fin_intent = norm_intent(intent) if intent else norm_intent(row.intent)
    if fin_intent not in INTENT_AR:
        raise ValueError(f"فئة غير معروفة: {fin_intent} | المسموح: {list(INTENT_AR)}")
    old_esc = 1 if to_bool(row.escalation) else 0
    fin_esc = (1 if to_bool(escalation) else 0) if escalation is not None else old_esc
    corrected = (fin_intent != norm_intent(row.intent)) or (fin_esc != old_esc)

    edited = edited_resolution.strip()
    final = edited or row.recommended_solution
    _set_review(rid, "معدّل" if (edited or corrected) else "معتمد", final, notes)
    if corrected:                                            # نصحّح التصنيف في السجل نفسه
        lg = load_log()
        lg.loc[lg.record_id == rid, ["intent", "escalation"]] = [fin_intent, str(fin_esc)]
        lg.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    entry = pd.DataFrame([{"call_id": rid, "intent": fin_intent, "escalation": fin_esc, "call_text": row.call_text,
                           "reference_transcript": row.call_text, "resolution": final}])
    entry.to_csv(APPROVED_PATH, mode="a", index=False, header=not os.path.exists(APPROVED_PATH), encoding="utf-8-sig")
    kb = pd.concat([kb, entry], ignore_index=True).fillna("")
    new_emb = emb_model.encode(row.call_text, convert_to_tensor=True, normalize_embeddings=True)
    kb_emb = torch.cat([kb_emb, new_emb.unsqueeze(0)])
    print(f"✓ {rid} انعتمد وانضاف للقاعدة | حجمها الحين: {len(kb)}")

def reject(rid, notes=""):
    _set_review(rid, "مرفوض", "", notes)
    print(f"✗ {rid} انرفض")

# ============ حذف وتعديل السجل (من الدفتر فقط، مو من الصفحة العامة) ============
BACKUP_DIR = os.path.join(STORE_DIR, "backups")

def backup_log(tag="manual"):
    """نسخة احتياطية من السجل والحلول المعتمدة قبل أي حذف أو تعديل. تنحفظ في CallSense/backups."""
    os.makedirs(BACKUP_DIR, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    saved = []
    for p in (LOG_PATH, APPROVED_PATH):
        if os.path.exists(p):
            dst = os.path.join(BACKUP_DIR, f"{tag}_{stamp}_{os.path.basename(p)}")
            shutil.copy(p, dst); saved.append(dst)
    print("💾 نسخة احتياطية:", [os.path.basename(x) for x in saved] or "ما فيه شي ينحفظ")
    return saved

def _remove_from_kb(rid):
    """لو المكالمة كانت معتمدة، نشيلها من القاعدة اللي يتعلم منها Qwen (الملف + الذاكرة)."""
    global kb, kb_emb
    if os.path.exists(APPROVED_PATH):
        ap = pd.read_csv(APPROVED_PATH, dtype=str).fillna("")
        if (ap.call_id == rid).any():
            ap[ap.call_id != rid].to_csv(APPROVED_PATH, index=False, encoding="utf-8-sig")
    keep = [i for i in range(len(kb)) if kb.call_id[i] != rid]
    if len(keep) != len(kb):
        kb = kb.iloc[keep].reset_index(drop=True)
        kb_emb = kb_emb[keep]
        print(f"   ↳ انشالت {rid} من قاعدة Qwen | حجمها الحين: {len(kb)}")

def delete_records(rids, backup=True):
    rids = [rids] if isinstance(rids, str) else list(rids)
    log = load_log()
    found = [r for r in rids if (log.record_id == r).any()]
    missing = [r for r in rids if r not in found]
    if missing:
        print("⚠️ ما لقيت:", missing)
    if not found:
        return
    if backup:
        backup_log("delete")
    log[~log.record_id.isin(found)].to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    for r in found:
        _remove_from_kb(r)
    print(f"🗑 انحذف {len(found)} سجل | المتبقي في السجل: {len(log) - len(found)}")

def delete_record(rid, backup=True):
    delete_records([rid], backup)

def delete_before(timestamp, backup=True):
    """يحذف كل المكالمات اللي قبل وقت معيّن، مفيد لتنظيف تجارب اليوم قبل العرض. مثال: delete_before("2026-11-01 14:00")"""
    ts = str(timestamp).replace(" ", "T")
    log = load_log()
    old = log[log.timestamp < ts].record_id.tolist()
    print(f"عدد السجلات اللي قبل {timestamp}: {len(old)}")
    if old:
        delete_records(old, backup)

def clear_log(confirm=False):
    """يمسح السجل كامل (مع نسخة احتياطية). لازم confirm=True. الحلول المعتمدة في approved_kb تبقى."""
    if not confirm:
        print("⚠️ هذا يمسح كل المكالمات في السجل. لو متأكدة: clear_log(confirm=True)")
        return
    backup_log("clear")
    pd.DataFrame(columns=LOG_COLS).to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    print("✓ السجل فاضي الحين")

EDITABLE = ["intent", "escalation", "summary", "solution_status", "applied_solution", "recommended_solution",
            "escalation_handoff", "final_resolution", "reviewer_notes", "call_text"]

def edit_record(rid, backup=True, **fields):
    """تعديل سجل. مثال: edit_record("R2026...", intent="technical", escalation=0, summary="...")"""
    bad = [k for k in fields if k not in EDITABLE]
    if bad:
        raise ValueError(f"حقول غير قابلة للتعديل: {bad} | المسموح: {EDITABLE}")
    log = load_log()
    m = log.record_id == rid
    if not m.any():
        raise ValueError(f"ما لقيت النتيجة {rid}")
    if "intent" in fields:
        fields["intent"] = norm_intent(fields["intent"])
        if fields["intent"] not in INTENT_AR:
            raise ValueError(f"فئة غير معروفة | المسموح: {list(INTENT_AR)}")
    if "escalation" in fields:
        fields["escalation"] = "1" if to_bool(fields["escalation"]) else "0"
    if backup:
        backup_log("edit")
    for k, v in fields.items():
        log.loc[m, k] = j(v)
    log.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    print(f"✓ تعدّل {rid}: {list(fields)}")
    if log[m].iloc[0].review_status in ("معتمد", "معدّل"):
        print("   ملاحظة: السجل معتمد قبل، والتعديل ما يغيّر نسخته داخل قاعدة Qwen. للتغيير الكامل: delete_record ثم أعيدي التحليل واعتمدي.")

## 6) الدالة الرئيسية: تربط الكل
صوت ← نص ← تصنيف وتصعيد ← ملخص وحل ← حفظ.

In [13]:
def run_pipeline(audio_path, save=True):
    """صوت ← نص (Whisper) ← تصنيف وتصعيد (Qwen) ← ملخص وحل (Qwen) ← حفظ في السجل."""
    t0 = time.time()
    transcript = transcribe_audio(audio_path)                                   # 1) Whisper
    if len(transcript.split()) < 3:
        return {"error": "ما قدرت أسمع كلام واضح في التسجيل. جرّبوا مرة ثانية وقرّبوا من المايك."}

    cls = classify_with_qwen(transcript)                                        # 2) Qwen: النطاق ثم الفئة والتصعيد
    res = generate_digest(transcript, cls["intent"], cls["escalation"],         # 3) Qwen: الملخص والحل
                          topic=cls["topic"], scope_confirmed=(cls["method"] == "qwen"))
    res["intent_reason"], res["intent_method"] = cls["reason"], cls["method"]
    rid = save_result(res, transcript) if save else None                        # حفظ في Drive

    return {"record_id": rid, "transcript": transcript, "classification": cls, "digest": res,
            "report": format_report(res), "seconds": round(time.time() - t0, 1)}

print("✅ run_pipeline() is ready")

✅ run_pipeline() is ready


## 7) تجربة بملف صوتي (اختياري)
حطوا تسجيل في مجلد `CallSense` باسم `test.m4a`.

In [14]:
TEST_AUDIO = os.path.join(STORE_DIR, "test.m4a")

if os.path.exists(TEST_AUDIO):
    out = run_pipeline(TEST_AUDIO, save=False)       # save=False عشان ما نلوّث السجل بتجربة
    if "error" in out:
        print(out["error"])
    else:
        print("⏱ الوقت:", out["seconds"], "ثانية\n")
        print("📝 النص:\n", out["transcript"], "\n")
        print("🏷 التصنيف:", out["classification"], "\n")
        print(out["report"])
else:
    print("ما لقيت", TEST_AUDIO, "، كمّلوا للخطوة 9")

⏱ الوقت: 75.1 ثانية

📝 النص:
 السلام عليكم ورحمة الله. أهلاً بك في خدمة العملاء اللي دا شركة الاتصالات. كيف أقدر خدمك اليوم؟ وعليكم السلام. أنا والله عندي استفسار أبو اسأل عن الألياف البصرية اللي تشبكونها في البيوت. كيف الطريقة؟ أهلاً فيك. عندنا خدمة الألياف البصرية. يكون عندك إنترنت متوفر في البيت. طول الوقت. ايش حاب؟ ايش رحلك بطبط؟ أبواك تفهمني كيف أقدر أرفع طلب ومكم تركبون لألياف في بيتي لأنه والله نت صغير هذا عددني خلاص وليهمك تقدر تدخل على التطبيق وتروح القسم الإنترنت القوي والدرب خيار الألياف البصرية تحجز موعد من هناك وبذلك لا يوصل الطلب الأجرب مورد لك ويجون يختمونك بتواصلهم معك على رجمك تمام يعطيك الفعافية والبضع سهلات ولا يأخذ وقت لاني والله مستعدل الموضوع إن شاء الله بيكون بسيط لكن ارفع طلب الان علشان يصلك بأقرب وقت تمام يعطيك العافية الله يعافيك أي خدمة أخرى لا شكرا 

🏷 التصنيف: {'intent': 'information', 'escalation': 0, 'reason': 'استفسار عن طريقة الحصول على خدمة الألياف البصرية', 'topic': 'استفسار عن الألياف البصرية', 'method': 'qwen', 'telecom_related': True, 'retried': Fa

## 7-ب) اختبار A/B لإعدادات Whisper (اختياري)
<div dir="rtl" style="text-align: right;">

يقارن إعداد Whisper الأصلي (`baseline`) بالمضبوط (`tuned`) على **نفس التسجيلات**، ويظهر هل الإعداد الأصلي يعطي نصوص مختلفة لنفس الصوت (عشوائية).

1. اضافة ٣–٥ تسجيلات في Drive داخل مجلد `CallSense/ab_tests/`
2. إزالة علامة `#` من آخر سطر ثم تشغيل الخلية
3. (اختياري) لو مكتوب النص الصحيح لتسجيل، بتمرر في `refs` وبيحسب نسبة الخطأ WER

يتم اختيار الأفضل لتسجيلاتكم من `WHISPER_MODE` في خلية Whisper.


</div>

In [ ]:
import glob, time

def _norm_ar(s):
    s = re.sub(r"[\u064B-\u0652\u0640]", "", str(s))                      # التشكيل والتطويل
    s = re.sub("[أإآ]", "ا", s).replace("ى", "ي").replace("ة", "ه")
    return re.sub(r"[^\w\s]", " ", s).split()

def wer(ref, hyp):
    """نسبة الأخطاء في الكلمات (أقل = أحسن). الصفر = مطابق تمامًا."""
    r, h = _norm_ar(ref), _norm_ar(hyp)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)] / max(len(r), 1)

def ab_test_whisper(files=None, refs=None, repeat_baseline=2):
    """
    يقارن Whisper الأصلي (baseline) بالمضبوط (tuned) على نفس التسجيلات.
    files: قائمة مسارات، أو None ليقرأ كل الملفات من مجلد CallSense/ab_tests في Drive
    refs : (اختياري) {"اسم_الملف.m4a": "النص الصحيح كما قيل"} عشان يحسب نسبة الخطأ WER
    """
    if files is None:
        files = sorted(f for f in glob.glob(os.path.join(STORE_DIR, "ab_tests", "*")) if os.path.isfile(f))
    assert files, "حطّوا ٣–٥ تسجيلات في مجلد CallSense/ab_tests في Drive ثم أعيدوا التشغيل"
    rows = []
    for f in files:
        name = os.path.basename(f)
        t0 = time.time(); base = [transcribe_audio(f, "baseline") for _ in range(repeat_baseline)]
        tb = (time.time() - t0) / repeat_baseline
        t0 = time.time(); tuned = transcribe_audio(f, "tuned"); tt = time.time() - t0
        print("=" * 70); print("📁", name)
        print(f"\n🔸 baseline ({tb:.1f}s):\n{base[0]}")
        if repeat_baseline > 1:
            print("   ↳ نفس الصوت مرتين بالإعداد الأصلي:", "نفس النص ✅" if len(set(base)) == 1 else "نصين مختلفين ⚠️ (عشوائية)")
        print(f"\n🔹 tuned ({tt:.1f}s):\n{tuned}\n")
        row = {"file": name, "baseline_stable": len(set(base)) == 1, "sec_baseline": round(tb, 1), "sec_tuned": round(tt, 1)}
        if refs and name in refs:
            row["wer_baseline"], row["wer_tuned"] = round(wer(refs[name], base[0]), 3), round(wer(refs[name], tuned), 3)
            print(f"📊 WER  baseline={row['wer_baseline']}  tuned={row['wer_tuned']}  (الأقل أحسن)\n")
        rows.append(row)
    res = pd.DataFrame(rows)
    if "wer_tuned" in res:
        print(f"المتوسط: baseline={res.wer_baseline.mean():.3f} | tuned={res.wer_tuned.mean():.3f}")
    return res

# ab = ab_test_whisper()                                                            # ← شيلي # وشغّليها
# ab = ab_test_whisper(refs={"call1.m4a": "النص الصحيح للتسجيل الأول ..."})         # ← مع النص الصحيح لحساب WER

## 8) (اختياري) قياس دقة تصنيف Qwen

يختبر Qwen على ٣٠ مكالمة من الداتا (٥ من كل فئة)، ويستبعد المكالمة نفسها من أمثلته، ويعطيكم **رقم حقيقي** وقائمة بالحالات اللي غلط فيها.

In [16]:
def evaluate_qwen_classifier(per_class=5, seed=0):
    """يختبر تصنيف Qwen على مكالمات من الداتا الأصلية (مع استبعاد المكالمة نفسها من الأمثلة)."""
    base = kb[kb.call_id.astype(str).str.startswith("CS")]
    sample = pd.concat([g.sample(min(per_class, len(g)), random_state=seed) for _, g in base.groupby("intent")])
    rows, t0 = [], time.time()
    for _, r in sample.iterrows():
        c = classify_with_qwen(r.call_text, call_id=r.call_id)
        rows.append({"call_id": r.call_id, "true_intent": norm_intent(r.intent), "pred_intent": c["intent"],
                     "true_esc": int(to_bool(r.escalation)), "pred_esc": c["escalation"],
                     "method": c["method"], "reason": c["reason"]})
    res = pd.DataFrame(rows)
    res["intent_ok"] = res.true_intent == res.pred_intent
    res["esc_ok"] = res.true_esc == res.pred_esc
    print(f"عدد المكالمات: {len(res)} | متوسط الوقت للمكالمة: {(time.time() - t0) / len(res):.1f} ثانية")
    print(f"دقة التصنيف:  {res.intent_ok.mean():.0%}")
    print(f"دقة التصعيد:  {res.esc_ok.mean():.0%}")
    print("\nدقة التصنيف لكل فئة:"); print(res.groupby("true_intent").intent_ok.mean().round(2).to_string())
    bad = res[~(res.intent_ok & res.esc_ok)]
    print(f"\nالحالات الغلط ({len(bad)}):")
    print(bad[["call_id", "true_intent", "pred_intent", "true_esc", "pred_esc", "reason"]].to_string(index=False))
    return res

eval_res = evaluate_qwen_classifier(per_class=5)    # ← شيلي # وشغّليها (٣٠ مكالمة تقريبًا)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


عدد المكالمات: 30 | متوسط الوقت للمكالمة: 16.0 ثانية
دقة التصنيف:  90%
دقة التصعيد:  87%

دقة التصنيف لكل فئة:
true_intent
billing         1.0
cancellation    0.6
complaint       1.0
information     1.0
out_of_scope    0.8
technical       1.0

الحالات الغلط (6):
call_id  true_intent  pred_intent  true_esc  pred_esc                                       reason
  CS035 cancellation cancellation         0         1        شكوى من تأخر إجراء الإلغاء ووجود رسوم
  CS030 cancellation  information         0         0 استفسار عن سعر وشروط الاشتراك في باقة إنترنت
  CS025 cancellation  information         1         0       استفسار عن متوفرية الخدمة في منزل جديد
  CS045    complaint    complaint         0         1 الموظف لم يشرح المشكلة وتكررت التجربة السيئة
  CS079 out_of_scope  information         0         0                       استفسار عن نصائح دولية
  CS018    technical    technical         0         1               مشكلة متكررة وانزعاج من الشركة


## 8-ب) حالات صعبة: خارج النطاق وغيرها (اختياري)
<div dir="rtl" style="text-align: right;">

١٣ مكالمة نصية فيها مطعم ومندوب توصيل وبنك وسؤال عام وحالات اتصالات صعبة. تشغيلها سريع (تصنيف فقط) وتعطي نسبة النجاح.



</div>

In [18]:
# حالات صعبة بصيغة نص Whisper (بدون صوت). النتيجة المتوقعة تقديرنا نحن، وأي اختلاف يستاهل نظرة مو بالضرورة غلط.
HARD_CASES = [
    ("مطعم: تأخر طلب", "out_of_scope", 0,
     "السلام عليكم حياك الله في خدمة العملاء. أنا طلبت وجبة من المطعم من ساعة ونص وللحين ما وصلت والمندوب ما يرد علي وأنا جوعان وزعلان مره. ابغى أعرف وين طلبي. نعتذر لك راح أتواصل مع المندوب وأشوف وين وصل."),
    ("مندوب توصيل وهدية", "out_of_scope", 0,
     "هلا والله أنا طلبت من التطبيق ستيكتو وعطيت المندوب هدية يوصلها لأختي لكن ما وصلت لها الهدية وهو مو راد علي. ابغى أحد يتواصل مع المندوب. أبشر نعتذر لك راح نرسل لك تحديث."),
    ("بنك: بطاقة موقوفة", "out_of_scope", 0,
     "السلام عليكم حياك الله في خدمة العملاء. بطاقتي الائتمانية انوقفت فجأة وما قدرت أدفع في المحل ابغى أعرف ليش وكيف أفعلها من جديد. نعتذر منك لازم تراجع فرع البنك."),
    ("سؤال عام (رياضة)", "out_of_scope", 0,
     "مرحبا ابغى أسألك مين فاز بكأس العالم السنة الماضية ومتى المباراة الجاية للهلال. أعتذر ما أقدر أساعد إلا في خدمات الاتصالات."),
    ("ترجمة رسالة", "out_of_scope", 0,
     "عندي رسالة بالإنجليزي ممكن تترجمها لي للعربي. أعتذر أنا أقدم خدمات الاتصالات فقط."),
    ("حجز موعد مستشفى", "out_of_scope", 0,
     "السلام عليكم ابغى أحجز موعد عند الدكتور في المستشفى بكرة الصبح. عذرا هذا مو من اختصاصنا."),
    ("فاتورة مضاعفة وعميل منزعج", "billing", 1,
     "الفاتورة وصلتني هذا الشهر مضاعفة ومو أول مرة كلمتكم الشهر الماضي ووعدوني يصححونها وما صار شي أنا تعبت. أبغى مسؤول يحل المشكلة الحين وإلا بلغي الاشتراك."),
    ("إنترنت ضعيف اليوم", "technical", 0,
     "الإنترنت عندي ضعيف اليوم بس وجربت أعيد تشغيل الراوتر. أبشر خلنا نشيك على الخط ونعيد ضبط الإعدادات من عندنا ونشوف."),
    ("استفسار باقات", "information", 0,
     "ابغى أعرف وش الباقات اللي عندكم للإنترنت المنزلي وكم سعر الباقة المفتوحة. أبشر عندنا عدة باقات حسب السرعة."),
    ("إلغاء بسبب مشاكل متكررة", "cancellation", 1,
     "ابغى ألغي اشتراكي لأن النت يفصل كل يوم وفتحت بلاغات أكثر من مرة وكلها تتقفل بدون حل تعبت منكم."),
    ("شكوى موظف فرع", "complaint", 0,
     "رحت الفرع أمس والموظف ما عنده تعاون ويرد علي بسرعة وبدون ما يشرح لي. أبغى أشتكي على أسلوبه."),
    ("موعد تركيب تأجل مرتين", "complaint", 1,
     "موعد تركيب الإنترنت تأجل مرتين وما أحد أبلغني ولا اتصل بي وأنا جالس أنتظر الفني. ابغى حل اليوم."),
    ("تحويل رصيد", "information", 0,
     "ابغى أحول رصيد لرقم ثاني كيف أسوي. أبشر تقدر تحوله من التطبيق أو بالكود."),
]

def run_hard_cases():
    """يشغّل تصنيف Qwen على الحالات الصعبة ويطبع النتيجة (تصنيف فقط، سريع)."""
    rows = []
    for name, exp_i, exp_e, text in HARD_CASES:
        c = classify_with_qwen(text)
        rows.append({"case": name, "expected": exp_i, "got": c["intent"], "esc_exp": exp_e, "esc_got": c["escalation"],
                     "ok": (c["intent"] == exp_i) and (c["escalation"] == exp_e), "intent_ok": c["intent"] == exp_i,
                     "method": c["method"], "reason": c["reason"]})
    r = pd.DataFrame(rows)
    print(r[["case", "expected", "got", "esc_exp", "esc_got", "ok"]].to_string(index=False))
    oos = r[r.expected == "out_of_scope"]
    print(f"\nالتصنيف صح: {r.intent_ok.sum()}/{len(r)} | الكل صح (تصنيف وتصعيد): {r.ok.sum()}/{len(r)}")
    print(f"خارج النطاق اتعرفت صح: {(oos.got == 'out_of_scope').sum()}/{len(oos)}")
    bad = r[~r.ok]
    if len(bad):
        print("\nللمراجعة:"); print(bad[["case", "got", "esc_got", "method", "reason"]].to_string(index=False))
    return r
hard = run_hard_cases()        # ← شيلي # وشغّليها (حوالي ١٣ تصنيف قصير)

                     case     expected          got  esc_exp  esc_got    ok
           مطعم: تأخر طلب out_of_scope out_of_scope        0        0  True
        مندوب توصيل وهدية out_of_scope out_of_scope        0        0  True
        بنك: بطاقة موقوفة out_of_scope out_of_scope        0        0  True
         سؤال عام (رياضة) out_of_scope out_of_scope        0        0  True
              ترجمة رسالة out_of_scope out_of_scope        0        0  True
          حجز موعد مستشفى out_of_scope out_of_scope        0        0  True
فاتورة مضاعفة وعميل منزعج      billing      billing        1        1  True
        إنترنت ضعيف اليوم    technical    technical        0        0  True
            استفسار باقات  information  information        0        0  True
  إلغاء بسبب مشاكل متكررة cancellation cancellation        1        1  True
            شكوى موظف فرع    complaint out_of_scope        0        0 False
    موعد تركيب تأجل مرتين    complaint    complaint        1        1  True
            

## 9) الموقع (Gradio) برابط عام 🚀



 الرابط الذي سيظهر  `https://....gradio.live`


- يشتغل **طول ما الدفتر شغال**، فلا يتم اغلاق الصفحة ولا الخلية. والرابط مؤقت ويتغير في كل تشغيل.
- الطلبات تنعالج **وحدة وحدة** على GPU واحد.

In [19]:
import html as _html

# ====== اختاري الألوان بتغيير هذا السطر: "navy" (كحلي + سماوي + لمسة خضراء) أو "slate" (رمادي + أزرق) ======
PALETTE = "navy"

PALETTES = {
    "navy":  dict(bg="#F3F7FB", ink="#0B1F3A", muted="#5B6B82", primary="#1D4E89", accent="#38BDF8",
                  mint="#2FBF9B", line="#D3E3F3", hero1="#0B1F3A", hero2="#1D4E89", sky1="#CFE8FA", sky2="#EEF7FD"),
    "slate": dict(bg="#F4F5F7", ink="#1F2933", muted="#66707D", primary="#2563EB", accent="#60A5FA",
                  mint="#5BB89A", line="#DADDE3", hero1="#2B3440", hero2="#3B4756", sky1="#DDE1E8", sky2="#F1F2F5"),
}

CSS_TEMPLATE = """
:root { --cs-bg:__BG__; --cs-ink:__INK__; --cs-muted:__MUTED__; --cs-primary:__PRIMARY__;
        --cs-accent:__ACCENT__; --cs-mint:__MINT__; --cs-line:__LINE__; }
html, body, gradio-app, .gradio-app { background: linear-gradient(180deg, __SKY1__ 0%, __SKY2__ 100%) fixed !important; min-height: 100vh; }
.gradio-container { max-width: 940px !important; margin: 0 auto !important; background: transparent !important; }
footer { display: none !important; }
.cs-hero { background: radial-gradient(620px 220px at 88% 0%, rgba(56,189,248,.30), transparent 70%), linear-gradient(135deg, __HERO1__, __HERO2__);
           border-radius: 16px; padding: 28px 28px 24px; margin: 4px 0 14px; border-bottom: 4px solid var(--cs-accent);
           display: flex; align-items: center; justify-content: space-between; gap: 20px; }
.cs-hero-text { flex: 1 1 auto; min-width: 0; }
.cs-wave { flex: none; width: 210px; height: 84px; }
.cs-pills { margin-top: 14px; display: flex; flex-wrap: wrap; gap: 8px; }
.cs-pills span { font-size: .8rem; padding: 4px 13px; border-radius: 999px; color: #DCEBFA;
                 border: 1px solid rgba(255,255,255,.28); background: rgba(255,255,255,.08); }
@media (max-width: 680px) { .cs-wave { display: none; } .cs-hero { padding: 22px 20px 20px; } }
.cs-hero h1 { margin: 0 0 10px; font-size: 1.65rem; font-weight: 700; line-height: 1.3; color: #FFFFFF; }
.cs-hero p  { margin: 0; font-size: 1rem; line-height: 1.6; color: #D9E8F7; }
.cs-note { font-size: .88rem; color: var(--cs-muted); padding: 2px 4px 6px; }
.cs-wrap { display: flex; flex-direction: column; gap: 12px; color: var(--cs-ink); line-height: 1.75; }
.cs-wrap p, .cs-wrap ol { margin: 0; }
.cs-wrap ol { padding-inline-start: 1.4rem; }
.cs-stage { background: #EAF4FD; border: 1px solid #C9DDF3; color: var(--cs-primary); border-radius: 12px;
            padding: 10px 14px; font-weight: 600; }
.cs-verdict { display: flex; flex-wrap: wrap; background: #FFFFFF; border: 1px solid var(--cs-line); border-radius: 14px; overflow: hidden; }
.cs-seg { flex: 1 1 260px; padding: 16px 18px; }
.cs-seg + .cs-seg { border-inline-start: 1px solid var(--cs-line); }
.cs-label { font-size: .88rem; color: var(--cs-muted); margin-bottom: 8px; }
.cs-chip { display: inline-block; padding: 7px 16px; border-radius: 999px; font-weight: 600; font-size: 1.02rem; border: 1px solid transparent; }
.cs-info  { background: #E6F0FB; color: #1D4E89; border-color: #C9DDF3; }
.cs-ok    { background: #E8F5EE; color: #2F7A5A; border-color: #C9E6D7; }
.cs-alert { background: #FCEDEC; color: #A5524D; border-color: #F3D2D0; }
.cs-warn  { background: #FFF6E2; color: #8F6A14; border-color: #F2E2B4; }
.cs-reason { color: var(--cs-muted); font-size: .93rem; padding: 0 6px; }
.cs-flags { background: #FFF9EC; border: 1px solid #F2E2B4; border-radius: 14px; padding: 14px 18px; }
.cs-flags .cs-title { color: #8F6A14; }
.cs-flags ul { margin: 6px 0 0; padding-inline-start: 1.3rem; }
.cs-panel { background: #FFFFFF; border: 1px solid var(--cs-line); border-radius: 14px; overflow: hidden; }
.cs-sec { padding: 16px 18px; }
.cs-sec + .cs-sec { border-top: 1px solid var(--cs-line); }
.cs-title { font-weight: 700; font-size: .98rem; color: var(--cs-primary); margin-bottom: 8px; }
.cs-title span { font-weight: 500; color: var(--cs-muted); }
.cs-handoff { background: #FFF7F6; }
.cs-reviewed { background: #F2FAF6; }
.cs-meta { display: flex; flex-wrap: wrap; gap: 8px 28px; padding: 0 6px; font-size: .93rem; color: var(--cs-muted); }
.cs-meta b { color: var(--cs-ink); font-weight: 600; }
.cs-trans { background: #FFFFFF; border: 1px solid var(--cs-line); border-radius: 14px; padding: 12px 18px; }
.cs-trans summary { cursor: pointer; font-weight: 700; color: var(--cs-primary); }
.cs-trans summary span { font-weight: 500; color: var(--cs-muted); }
.cs-cap { font-size: .85rem; color: var(--cs-muted); margin: 6px 0; }
.cs-empty { text-align: center; color: var(--cs-muted); padding: 34px 12px; border: 1px dashed var(--cs-line); border-radius: 14px; background: #FFFFFF; }
.cs-stats { color: var(--cs-muted); padding: 2px 6px 8px; }
.cs-stats b { color: var(--cs-ink); }
.cs-footer { text-align: center; color: var(--cs-muted); font-size: .88rem; line-height: 1.9; padding: 22px 8px 6px; margin-top: 10px; border-top: 1px solid var(--cs-line); }
.cs-footer b { color: var(--cs-ink); }
"""

def build_css(name=None):
    p = PALETTES[name or PALETTE]
    css = CSS_TEMPLATE
    for k, v in p.items():
        css = css.replace(f"__{k.upper()}__", v)
    return css

def _wave_svg(n=27):
    hs = [10, 22, 34, 20, 46, 30, 62, 38, 54, 26, 70, 44, 58, 32, 76, 48, 36, 64, 28, 52, 40, 22, 34, 16, 26, 12, 8]
    bars = "".join(f'<rect x="{i * 7.6:.1f}" y="{(84 - h) / 2:.1f}" width="4" height="{h}" rx="2" '
                   f'fill="#7DD3FC" opacity="{0.35 + 0.65 * min(h, 70) / 70:.2f}"/>' for i, h in enumerate(hs[:n]))
    return f'<svg class="cs-wave" viewBox="0 0 206 84" aria-hidden="true">{bars}</svg>'

HERO_HTML = ('<div class="cs-hero"><div class="cs-hero-text">'
             '<h1>📞 CallSense: Arabic Telecom Call-Center Intelligence!</h1>'
             '<p>Less manual listening. Earlier escalation signals. Quick, simple responses with reusable resolutions.</p>'
             '<div class="cs-pills"><span>Speech to Text</span><span>Intent &amp; Escalation</span><span>Summary &amp; Resolution</span></div>'
             f'</div>{_wave_svg()}</div>')

FOOTER_HTML = ('<div class="cs-footer"><b>CallSense by Zenith Team</b><br>'
               'Layan Almashhour • Ghadi Albishri • Leen Alahmary<br>'
               'Under the AI Engineering Bootcamp (AIEP) - OLO ACADEMY</div>')

NOTE_HTML = ('<div class="cs-note">Demo only: please do not record real customer data | '
             'للتجربة فقط، لا يتم تسجيل بيانات عملاء حقيقية.</div>')

EMPTY_HTML = ('<div class="cs-empty">Your analysis will appear here | يظهر التحليل هنا بعد التسجيل</div>')
PICK_HTML  = ('<div class="cs-empty">Select a row to see the full report | اضغطوا على أي صف لعرض التقرير الكامل</div>')

# ---------------- ثنائية اللغة ----------------
INTENT_EN = {"billing": "Billing", "technical": "Technical", "cancellation": "Cancellation",
             "complaint": "Complaint", "information": "Information", "out_of_scope": "Out of Scope"}
STATUS_EN = {"واضح": "Clear", "جزئي": "Partial", "غير موجود": "Not found", "خارج النطاق": "Out of scope"}
REVIEW_EN = {"معتمد": "Approved", "معدّل": "Edited", "مرفوض": "Rejected", "": "Pending review"}
REVIEW_AR = {"معتمد": "معتمد", "معدّل": "معدّل", "مرفوض": "مرفوض", "": "بانتظار المراجعة"}
FLAG_EN = [
    ("الموديل ما رجع JSON صحيح", "Model did not return valid JSON"),
    ("استشهد بمكالمات غير مسترجعة", "Cited calls that were not retrieved"),
    ("الموديل يشك في الفئة", "Model doubts the intent classification"),
    ("التصعيد مطلوب لكن ما فيه ملاحظة تسليم", "Escalation required but no handoff note"),
    ("التصنيف خارج النطاق لكن المكالمات المشابهة", "Marked out of scope, but similar calls are telecom-related"),
    ("المصنّف قال خارج النطاق لكن المكالمات المشابهة", "Marked out of scope, but similar calls are telecom-related"),
    ("المكالمات المشابهة تشير إلى أن الطلب قد يكون خارج النطاق", "Similar calls suggest the request may be out of scope"),
    ("مخرجات الموديل لم تكن بالعربية السليمة", "Model output was not clean Arabic"),
    ("لم يُولَّد حل واضح", "No clear resolution was generated"),
]

def _e(x):
    return _html.escape("" if x is None else str(x))

def _br(x):
    return _e(x).replace("\n", "<br>")

def intent_label(k):
    k = str(k or "")
    return f"{INTENT_EN.get(k, k)} | {INTENT_AR.get(k, k)}" if k else ""

def esc_label(flag):
    return "Required | مطلوب 🚨" if flag else "Not required | غير مطلوب ✓"

def flag_label(f):
    for ar, en in FLAG_EN:
        if ar in f:
            return f"{en} | {f}"
    return f

def _lines(v):
    if isinstance(v, list):
        return [str(x) for x in v if str(x).strip()]
    return [x for x in str(v or "").split("\n") if x.strip()]

def _sec(title_en, title_ar, body, extra=""):
    return f'<div class="cs-sec {extra}"><div class="cs-title">{_e(title_en)} <span>| {_e(title_ar)}</span></div>{body}</div>'

def render_result_html(v, stage=None):
    """v: dict فيه ما توفّر من النتائج (transcript, intent, escalation, summary, ...). stage: رسالة التقدم."""
    out = []
    if stage:
        out.append(f'<div class="cs-stage">⏳ {_e(stage)}</div>')

    if v.get("intent"):
        esc = bool(v.get("escalation"))
        out.append(
            '<div class="cs-verdict">'
            f'<div class="cs-seg"><div class="cs-label">Intent Classification | التصنيف</div>'
            f'<span class="cs-chip cs-info">{_e(intent_label(v["intent"]))}</span></div>'
            f'<div class="cs-seg"><div class="cs-label">Escalation | التصعيد</div>'
            f'<span class="cs-chip {"cs-alert" if esc else "cs-ok"}">{_e(esc_label(esc))}</span></div></div>')
        if v.get("reason"):
            out.append(f'<div class="cs-reason" dir="auto">Why | السبب: {_e(v["reason"])}</div>')

    flags = _lines(v.get("flags"))
    if flags:
        li = "".join(f'<li dir="auto">{_e(flag_label(f))}</li>' for f in flags)
        out.append(f'<div class="cs-flags"><div class="cs-title">⚠️ Review Needed <span>| يحتاج مراجعة</span></div><ul>{li}</ul></div>')

    secs = []
    if v.get("summary"):
        secs.append(_sec("Summary", "الملخص", f'<p dir="auto">{_br(v["summary"])}</p>'))
    if v.get("solution_status") == "خارج النطاق":
        secs.append(_sec("Response", "الرد", f'<p dir="auto">{_br(OOS_REPLY)}</p>'))
    elif v.get("solution_status") or v.get("recommended_solution"):
        st = v.get("solution_status", "")
        chip = f'<span class="cs-chip cs-warn" style="font-size:.85rem;padding:2px 10px;margin-bottom:8px">{_e(STATUS_EN.get(st, st))} | {_e(st)}</span><br>' if st else ""
        steps = "".join(f'<li dir="auto">{_e(strip_num(s))}</li>' for s in _lines(v.get("recommended_solution")))
        secs.append(_sec("Recommended Resolution", "الحل المقترح", f'{chip}<ol>{steps}</ol>'))
    if v.get("applied_solution"):
        secs.append(_sec("Applied by the Agent", "ما طبّقه الموظف", f'<p dir="auto">{_br(v["applied_solution"])}</p>'))
    if v.get("escalation_handoff"):
        secs.append(_sec("Escalation Handoff", "ملاحظة التسليم للمشرف", f'<p dir="auto">{_br(v["escalation_handoff"])}</p>', "cs-handoff"))
    if v.get("review_status") in ("معتمد", "معدّل") and v.get("final_resolution"):
        secs.append(_sec("Final Reviewed Resolution", "الحل النهائي المعتمد", f'<p dir="auto">{_br(v["final_resolution"])}</p>', "cs-reviewed"))
    if secs:
        out.append(f'<div class="cs-panel">{"".join(secs)}</div>')

    meta = []
    src_ids = norm_ids(_lines(v.get("sources")))
    if src_ids:
        meta.append(f'<span>Based on calls | مبني على مكالمات: <b dir="ltr">{_e(", ".join(src_ids))}</b></span>')
    if "review_status" in v:
        r = v.get("review_status", "")
        meta.append(f'<span>Review | المراجعة: <b>{_e(REVIEW_EN.get(r, r))} | {_e(REVIEW_AR.get(r, r))}</b></span>')
    if meta:
        out.append(f'<div class="cs-meta">{"".join(meta)}</div>')

    if v.get("transcript"):
        opened = " open" if (stage and not v.get("intent")) else ""
        out.append(f'<details class="cs-trans"{opened}><summary>Call Transcript <span>| نص المكالمة</span></summary>'
                   f'<div class="cs-cap">Transcribed by Whisper small</div><p dir="auto">{_br(v["transcript"])}</p></details>')

    return f'<div class="cs-wrap">{"".join(out)}</div>'

# ---------------- جدول السجل ----------------
H_ID, H_TIME, H_INT, H_ESC = "ID", "Time | الوقت", "Intent | الفئة", "Escalation | التصعيد"
H_SUM, H_RES, H_REV = "Summary | الملخص", "Resolution | حالة الحل", "Review | المراجعة"
HIST_COLS = [H_ID, H_TIME, H_INT, H_ESC, H_SUM, H_RES, H_REV]

def _is1(v):
    return str(v).strip() in ("1", "1.0", "True", "true")

def load_history(n=50):
    log = load_log()
    if log.empty:
        return pd.DataFrame(columns=HIST_COLS)
    h = log.tail(n).iloc[::-1]
    return pd.DataFrame({
        H_ID: h.record_id.values,
        H_TIME: h.timestamp.str.replace("T", " ").values,
        H_INT: [intent_label(x) for x in h.intent],
        H_ESC: ["Yes | نعم" if _is1(x) else "No | لا" for x in h.escalation],
        H_SUM: h.summary.values,
        H_RES: [f"{STATUS_EN.get(s, s)} | {s}" if s else "" for s in h.solution_status],
        H_REV: [f"{REVIEW_EN.get(r, r)} | {REVIEW_AR.get(r, r)}" for r in h.review_status],
    })

def log_stats_html():
    log = load_log()
    n = len(log)
    esc = int(sum(_is1(x) for x in log.escalation)) if n else 0
    return (f'<div class="cs-stats"><b>{n}</b> calls logged | مكالمة مسجّلة &nbsp;&nbsp; '
            f'<b>{esc}</b> escalated | تصعيد</div>')

def row_to_view(r):
    return {"transcript": r.call_text, "intent": r.intent, "escalation": _is1(r.escalation), "reason": r.intent_reason,
            "summary": r.summary, "solution_status": r.solution_status, "recommended_solution": _lines(r.recommended_solution),
            "applied_solution": r.applied_solution, "escalation_handoff": r.escalation_handoff,
            "sources": _lines(r.sources), "confidence": r.confidence, "flags": _lines(r.flags),
            "review_status": r.review_status, "final_resolution": r.final_resolution}

print("✅ ui helpers جاهزة | الألوان:", PALETTE)

✅ ui helpers جاهزة | الألوان: navy


In [ ]:
import gradio as gr

THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.blue, secondary_hue=gr.themes.colors.sky, neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("IBM Plex Sans Arabic"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill=f"linear-gradient(180deg, {PALETTES[PALETTE]['sky1']} 0%, {PALETTES[PALETTE]['sky2']} 100%)",
    body_background_fill_dark=f"linear-gradient(180deg, {PALETTES[PALETTE]['sky1']} 0%, {PALETTES[PALETTE]['sky2']} 100%)",
)
FORCE_LIGHT_JS = "() => { document.body.classList.remove('dark'); }"     # نثبّت الوضع الفاتح عشان الألوان تطلع مثل ما صممناها

def analyze(audio_path):
    if not audio_path:
        raise gr.Error("Please record or upload a call first | سجّلوا مكالمة أول (أو ارفعوا ملف صوتي).")
    t0 = time.time()
    try:
        yield render_result_html({}, stage="Step 1/3: Transcribing audio | تحويل الصوت إلى نص"), ""
        transcript = transcribe_audio(audio_path)
        if len(transcript.split()) < 3:
            raise gr.Error("Could not hear clear speech, please try again closer to the mic | "
                           "ما قدرت أسمع كلام واضح، جرّبوا مرة ثانية وقرّبوا من المايك.")
        v = {"transcript": transcript}
        yield render_result_html(v, stage="Step 2/3: Classifying intent and escalation | تصنيف المكالمة وتحديد التصعيد"), ""

        cls = classify_with_qwen(transcript)
        v.update(intent=cls["intent"], escalation=bool(cls["escalation"]), reason=cls["reason"])
        yield render_result_html(v, stage="Step 3/3: Writing summary and resolution | كتابة الملخص والحل"), ""

        res = generate_digest(transcript, cls["intent"], cls["escalation"], topic=cls["topic"], scope_confirmed=(cls["method"] == "qwen"))
        res["intent_reason"], res["intent_method"] = cls["reason"], cls["method"]
        rid = save_result(res, transcript)
        v.update(res)
        v.update(escalation=bool(res["escalation"]), reason=cls["reason"], review_status="")
        foot = f"Record ID | رقم السجل: {rid} &nbsp;&nbsp; Processing time | وقت المعالجة: {round(time.time() - t0, 1)} s"
        yield render_result_html(v), foot
    except gr.Error:
        raise
    except Exception as ex:
        raise gr.Error(f"Something went wrong | حدث خطأ: {type(ex).__name__}: {str(ex)[:200]}")

def refresh_log():
    return log_stats_html(), load_history()

def show_details(df, evt: gr.SelectData):
    try:
        rid = df.iloc[evt.index[0]][H_ID]
        m = load_log()
        m = m[m.record_id == rid]
        return render_result_html(row_to_view(m.iloc[0])) if len(m) else PICK_HTML
    except Exception:
        return PICK_HTML

with gr.Blocks(title="CallSense", theme=THEME, css=build_css(), js=FORCE_LIGHT_JS) as demo:
    gr.HTML(HERO_HTML)
    with gr.Tab("Call Analysis"):
        audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎙 Record the call | تسجيل المكالمة")
        gr.HTML(NOTE_HTML)
        btn = gr.Button("Analyze Call | حلّل المكالمة", variant="primary")
        result = gr.HTML(EMPTY_HTML)
        foot = gr.Markdown()
    with gr.Tab("Call Log"):
        stats = gr.HTML(log_stats_html)
        refresh = gr.Button("Refresh | تحديث")
        table = gr.Dataframe(value=load_history, interactive=False, wrap=True)
        detail = gr.HTML(PICK_HTML)
    gr.HTML(FOOTER_HTML)

    run = btn.click(analyze, inputs=audio, outputs=[result, foot], concurrency_limit=1)
    run.then(refresh_log, None, [stats, table])
    refresh.click(refresh_log, None, [stats, table])
    table.select(show_details, inputs=table, outputs=detail)
    if hasattr(gr, "Timer"):                                  # تحديث تلقائي للسجل كل ٢٠ ثانية
        gr.Timer(20).tick(refresh_log, None, [stats, table])

demo.queue(max_size=10)
demo.launch(share=True, debug=True, show_api=False)

## 10) (للفريق) مراجعة واعتماد الحلول
الموقع العام ما فيه أزرار اعتماد **عن قصد**: أي أحد معه الرابط يقدر يضغط، والاعتماد يضيف للقاعدة اللي يتعلم منها Qwen. أوقفوا خلية الموقع أول، ثم:
```python
display(pending())                                        # النتائج اللي ما انراجعت
approve("R2026...")                                       # اعتماد كما هو
approve("R2026...", edited_resolution="1. ...\n2. ...")   # اعتماد مع تعديل الحل
approve("R2026...", intent="complaint", escalation=1)     # تصحيح تصنيف Qwen أو تصعيده، ثم اعتماد
reject("R2026...", notes="الحل غير دقيق")
```
تصحيح التصنيف مهم: المكالمة المعتمدة تدخل القاعدة بتصنيفها، وQwen يستخدمها كمثال. فلا نبي تصنيف غلط يدخل القاعدة.

### حذف وتعديل السجل
كل هذي من الدفتر فقط، وتاخذ **نسخة احتياطية تلقائية** في `CallSense/backups` قبل أي تغيير:
```python
delete_record("R2026...")                       # حذف مكالمة (وتنشال من قاعدة Qwen لو كانت معتمدة)
delete_records(["R2026...", "R2026..."])        # حذف عدة مكالمات
delete_before("2026-11-01 14:00")               # تنظيف كل تجارب اليوم قبل وقت معيّن (قبل العرض)
edit_record("R2026...", intent="technical", escalation=0, summary="...")   # تعديل حقول
clear_log(confirm=True)                         # مسح السجل كامل (مع نسخة احتياطية)
backup_log()                                    # نسخة احتياطية يدوية
```